# Ranker blend

Я смешиваю ранги двух LightGBM seed, XGBoost и стабильного LightGBM через reciprocal rank fusion.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path(".")
DATA_DIR = ROOT / "statement" / "candidate_public" / "candidate_data"
sys.path.insert(0, str(ROOT / "src"))
results = pd.read_csv(ROOT / "outputs" / "experiment_results.csv")

In [2]:
results[results["stage"].isin(["Ranker", "Blend", "Final"])].sort_values("MAP@10")

,stage,approach,MAP@10,validation
5,Ranker,LambdaMART base,0.674602,nested 5-fold OOF
6,Ranker,XGBoost pairwise,0.694662,nested 5-fold OOF
7,Ranker,LightGBM bagged,0.697418,nested 5-fold OOF
8,Blend,LGBM + XGBoost,0.698829,OOF blend
9,Final,Ranker RRF ensemble,0.699566,OOF blend


In [3]:
weights = {
    "lgbm_seed_42": 0.4644,
    "lgbm_seed_123": 0.0108,
    "xgboost_pairwise": 0.4767,
    "lgbm_stable": 0.0481,
}
weights

{'lgbm_seed_42': 0.4644,
 'lgbm_seed_123': 0.0108,
 'xgboost_pairwise': 0.4767,
 'lgbm_stable': 0.0481}

Я подобрал веса 0.4644, 0.0108, 0.4767 и 0.0481 по сохранённым OOF-предсказаниям. Эта комбинация дала лучший OOF MAP@10. Я смешиваю ранги, а не raw-score, потому что шкалы LightGBM и XGBoost различаются.

> Для сохранения submission я раскомментирую и выполняю следующую клетку.

In [4]:
# from pipeline import fit_predict

# answer = fit_predict(DATA_DIR)
# OUTPUT_PATH = ROOT / "outputs" / "answer.csv"
# answer.to_csv(OUTPUT_PATH, index=False)
# OUTPUT_PATH